# Treinamento NLP local e regularizado

Classificador multirrótulo para:

- `pedido_socorro`
- `ameaca`
- `agressao`
- `referencia_arma`
- `nao_violento`

Esta versão foi preparada para execução local no projeto `violence-edge-ai` e inclui:

- caminhos locais automáticos;
- salvamento por checkpoint;
- retomada após interrupção;
- menos épocas;
- dropout maior;
- congelamento das camadas inferiores do BERTimbau;
- suavização leve dos rótulos;
- auditoria contra vazamento entre os splits;
- correção da escolha dos limiares;
- avaliação opcional em `teste_externo.jsonl`;
- ZIP final sem os checkpoints pesados.

> As regularizações reduzem sobreajuste, mas não substituem um teste externo com transcrições reais do Whisper.


## Antes de executar

Inicie o Jupyter dentro da raiz do projeto:

```bash
cd ~/Documentos/projeto-ia-embarcada/violence-edge-ai
jupyter lab
```

Os arquivos precisam estar em:

```text
datasets/nlp/splits/
├── treino.jsonl
├── validacao.jsonl
├── teste.jsonl
└── teste_externo.jsonl   # opcional
```

Execute as células em ordem. O modelo e as métricas são gravados diretamente no computador.


In [ ]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [ ]:
from dataclasses import asdict, dataclass
from pathlib import Path
import json
import random
import re
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    hamming_loss,
    multilabel_confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm.auto import tqdm
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
print("CUDA do PyTorch:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    memoria_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {memoria_gb:.2f} GB")
else:
    print("AVISO: o treinamento será executado na CPU e poderá demorar bastante.")


## 1. Configuração e caminhos


In [ ]:
@dataclass(frozen=True)
class Configuracao:
    modelo_base: str = "neuralmind/bert-base-portuguese-cased"
    comprimento_maximo: int = 64

    epocas: int = 2
    lote_treino: int = 4
    lote_avaliacao: int = 8
    acumulacao_gradientes: int = 4

    taxa_aprendizado: float = 1e-5
    decaimento_peso: float = 0.05
    aquecimento: float = 0.10
    maximo_gradiente: float = 1.0

    dropout_oculto: float = 0.20
    dropout_atencao: float = 0.20
    dropout_classificador: float = 0.30
    camadas_iniciais_congeladas: int = 8
    suavizacao_rotulos: float = 0.02

    paciencia: int = 1
    passos_log: int = 100
    semente: int = 42

    limiar_inicial: float = 0.50
    limiar_minimo: float = 0.10
    limiar_maximo: float = 0.90
    passo_limiar: float = 0.01

    retomar_checkpoint: bool = True


configuracao = Configuracao()

ROTULOS = [
    "pedido_socorro",
    "ameaca",
    "agressao",
    "referencia_arma",
    "nao_violento",
]

ID_PARA_ROTULO = {
    indice: rotulo
    for indice, rotulo in enumerate(ROTULOS)
}

ROTULO_PARA_ID = {
    rotulo: indice
    for indice, rotulo in ID_PARA_ROTULO.items()
}


def localizar_raiz_projeto() -> Path:
    candidatos = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path.home() / "Documentos" / "projeto-ia-embarcada" / "violence-edge-ai",
        Path.home() / "Documentos" / "ia_embarcada" / "violence-edge-ai",
        Path.home() / "Documentos" / "violence-edge-ai",
    ]

    for candidato in candidatos:
        candidato = candidato.expanduser().resolve()
        pasta_splits = candidato / "datasets" / "nlp" / "splits"

        if pasta_splits.exists():
            return candidato

    raise FileNotFoundError(
        "Não encontrei a raiz do projeto. Inicie o Jupyter dentro de "
        "violence-edge-ai ou adicione o caminho em localizar_raiz_projeto()."
    )


PASTA_RAIZ = localizar_raiz_projeto()
PASTA_DADOS = PASTA_RAIZ / "datasets" / "nlp" / "splits"
PASTA_CHECKPOINTS = (
    PASTA_RAIZ
    / "models"
    / "nlp"
    / "checkpoints_bertimbau_regularizado"
)
PASTA_MODELO = (
    PASTA_RAIZ
    / "models"
    / "nlp"
    / "bertimbau_multirrotulo_regularizado"
)
PASTA_METRICAS = (
    PASTA_RAIZ
    / "outputs"
    / "metrics"
    / "nlp"
    / "bertimbau_regularizado"
)
PASTA_RESULTADOS = PASTA_RAIZ / "outputs" / "resultados_nlp"

for pasta in (
    PASTA_CHECKPOINTS,
    PASTA_MODELO,
    PASTA_METRICAS,
    PASTA_RESULTADOS,
):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(configuracao.semente)
np.random.seed(configuracao.semente)
torch.manual_seed(configuracao.semente)
set_seed(configuracao.semente)

print("Raiz do projeto:", PASTA_RAIZ)
print("Dados:", PASTA_DADOS)
print("Checkpoints:", PASTA_CHECKPOINTS)
print("Modelo final:", PASTA_MODELO)
print("Métricas:", PASTA_METRICAS)

display(pd.DataFrame([asdict(configuracao)]).transpose())


## 2. Carregar e validar os splits


In [ ]:
def localizar_arquivo(nome: str, obrigatorio: bool = True) -> Path | None:
    encontrados = sorted(PASTA_DADOS.rglob(nome))

    if encontrados:
        return encontrados[0]

    if obrigatorio:
        raise FileNotFoundError(
            f"Arquivo não encontrado: {nome} em {PASTA_DADOS}"
        )

    return None


def carregar_jsonl(caminho: Path, nome_split: str) -> pd.DataFrame:
    registros = []

    with caminho.open("r", encoding="utf-8") as arquivo:
        for numero_linha, linha in enumerate(
            tqdm(
                arquivo,
                desc=f"Carregando {nome_split}",
                unit="linha",
            ),
            start=1,
        ):
            linha = linha.strip()

            if not linha:
                continue

            try:
                registro = json.loads(linha)
            except json.JSONDecodeError as erro:
                raise ValueError(
                    f"JSON inválido em {caminho}, linha {numero_linha}."
                ) from erro

            registros.append(registro)

    tabela = pd.DataFrame(registros)
    campos_obrigatorios = {"id", "texto", "rotulos"}
    faltantes = campos_obrigatorios - set(tabela.columns)

    if faltantes:
        raise ValueError(
            f"Campos ausentes em {nome_split}: {sorted(faltantes)}"
        )

    if tabela.empty:
        raise ValueError(f"O split {nome_split} está vazio.")

    tabela["split"] = nome_split
    tabela["texto"] = tabela["texto"].astype(str).str.strip()

    if (tabela["texto"] == "").any():
        raise ValueError(f"Existem textos vazios em {nome_split}.")

    return tabela


arquivos = {
    "treino": localizar_arquivo("treino.jsonl"),
    "validacao": localizar_arquivo("validacao.jsonl"),
    "teste": localizar_arquivo("teste.jsonl"),
}

arquivo_externo = localizar_arquivo(
    "teste_externo.jsonl",
    obrigatorio=False,
)

if arquivo_externo is not None:
    arquivos["teste_externo"] = arquivo_externo

for nome, caminho in arquivos.items():
    print(nome, "→", caminho)


tabelas = {
    nome: carregar_jsonl(caminho, nome)
    for nome, caminho in arquivos.items()
}

resumo_splits = pd.DataFrame(
    [
        {
            "split": nome,
            "exemplos": len(tabela),
        }
        for nome, tabela in tabelas.items()
    ]
)

display(resumo_splits)
display(tabelas["treino"].head())


## 3. Auditoria contra duplicação e vazamento


In [ ]:
def normalizar_texto(texto: str) -> str:
    texto = texto.lower().strip()
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"[^\w\s]", "", texto, flags=re.UNICODE)
    return texto


for tabela in tabelas.values():
    tabela["texto_normalizado"] = tabela["texto"].apply(
        normalizar_texto
    )


def contar_intersecao(
    tabela_a: pd.DataFrame,
    tabela_b: pd.DataFrame,
    coluna: str,
) -> int:
    valores_a = set(tabela_a[coluna].dropna().astype(str))
    valores_b = set(tabela_b[coluna].dropna().astype(str))
    return len(valores_a & valores_b)


linhas_auditoria = []
splits_principais = ["treino", "validacao", "teste"]

for indice, split_a in enumerate(splits_principais):
    for split_b in splits_principais[indice + 1:]:
        for coluna in ("id", "texto_normalizado"):
            linhas_auditoria.append(
                {
                    "split_a": split_a,
                    "split_b": split_b,
                    "criterio": coluna,
                    "intersecoes": contar_intersecao(
                        tabelas[split_a],
                        tabelas[split_b],
                        coluna,
                    ),
                }
            )

        if all(
            "grupo_id" in tabelas[split].columns
            for split in (split_a, split_b)
        ):
            linhas_auditoria.append(
                {
                    "split_a": split_a,
                    "split_b": split_b,
                    "criterio": "grupo_id",
                    "intersecoes": contar_intersecao(
                        tabelas[split_a],
                        tabelas[split_b],
                        "grupo_id",
                    ),
                }
            )


auditoria = pd.DataFrame(linhas_auditoria)
display(auditoria)

auditoria.to_csv(
    PASTA_METRICAS / "auditoria_vazamento.csv",
    index=False,
)

if (auditoria["intersecoes"] > 0).any():
    raise ValueError(
        "Foi detectada sobreposição entre treino, validação e teste. "
        "Corrija os splits antes de treinar."
    )

print("Auditoria concluída: nenhuma sobreposição detectada.")


## 4. Preparar os vetores multirrótulo


In [ ]:
def criar_vetor_rotulos(rotulos: dict) -> list[float]:
    if not isinstance(rotulos, dict):
        raise ValueError("O campo 'rotulos' precisa ser um dicionário.")

    vetor = []

    for nome in ROTULOS:
        valor = rotulos.get(nome)

        if valor not in (0, 1):
            raise ValueError(
                f"Rótulo inválido para {nome}: {valor}"
            )

        vetor.append(float(valor))

    indice_normal = ROTULO_PARA_ID["nao_violento"]

    if vetor[indice_normal] == 1.0 and sum(vetor) != 1.0:
        raise ValueError(
            "O rótulo nao_violento precisa ser exclusivo."
        )

    if sum(vetor) == 0.0:
        raise ValueError("Exemplo sem nenhum rótulo ativo.")

    return vetor


for nome, tabela in tabelas.items():
    tabela["labels"] = tabela["rotulos"].apply(
        criar_vetor_rotulos
    )

    matriz = np.asarray(
        tabela["labels"].tolist(),
        dtype=np.float32,
    )

    print(f"\n{nome}:")
    display(
        pd.Series(
            matriz.sum(axis=0).astype(int),
            index=ROTULOS,
            name="positivos",
        ).to_frame()
    )


## 5. Pesos das classes


In [ ]:
rotulos_treino = np.asarray(
    tabelas["treino"]["labels"].tolist(),
    dtype=np.float32,
)

positivos = rotulos_treino.sum(axis=0)
negativos = len(rotulos_treino) - positivos

pesos_positivos = negativos / np.maximum(positivos, 1)
pesos_positivos = np.clip(pesos_positivos, 0.5, 5.0)

peso_tabela = pd.DataFrame(
    {
        "rotulo": ROTULOS,
        "positivos": positivos.astype(int),
        "negativos": negativos.astype(int),
        "peso_positivo": pesos_positivos,
    }
)

display(peso_tabela)

tensor_pesos_positivos = torch.tensor(
    pesos_positivos,
    dtype=torch.float32,
)


## 6. Tokenizador e datasets


In [ ]:
tokenizador = AutoTokenizer.from_pretrained(
    configuracao.modelo_base,
    do_lower_case=False,
)


def criar_dataset(tabela: pd.DataFrame) -> Dataset:
    dados = tabela[["texto", "labels"]].copy()

    return Dataset.from_pandas(
        dados,
        preserve_index=False,
    )


def tokenizar_lote(lote: dict) -> dict:
    return tokenizador(
        lote["texto"],
        truncation=True,
        max_length=configuracao.comprimento_maximo,
    )


datasets = {
    nome: criar_dataset(tabela)
    for nome, tabela in tabelas.items()
}

datasets_tokenizados = {
    nome: dataset.map(
        tokenizar_lote,
        batched=True,
        desc=f"Tokenizando {nome}",
    )
    for nome, dataset in datasets.items()
}

colator = DataCollatorWithPadding(
    tokenizer=tokenizador,
)

print(datasets_tokenizados["treino"])
display(datasets_tokenizados["treino"][0])


## 7. Criar o BERTimbau regularizado


In [ ]:
configuracao_modelo = AutoConfig.from_pretrained(
    configuracao.modelo_base,
    num_labels=len(ROTULOS),
    id2label=ID_PARA_ROTULO,
    label2id=ROTULO_PARA_ID,
    problem_type="multi_label_classification",
)

configuracao_modelo.hidden_dropout_prob = configuracao.dropout_oculto
configuracao_modelo.attention_probs_dropout_prob = configuracao.dropout_atencao
configuracao_modelo.classifier_dropout = configuracao.dropout_classificador

modelo = AutoModelForSequenceClassification.from_pretrained(
    configuracao.modelo_base,
    config=configuracao_modelo,
)

if not hasattr(modelo, "bert"):
    raise AttributeError(
        "O modelo carregado não possui o atributo 'bert'. "
        "Este notebook foi preparado para o BERTimbau."
    )

for parametro in modelo.bert.embeddings.parameters():
    parametro.requires_grad = False

quantidade_camadas = len(modelo.bert.encoder.layer)
quantidade_congelada = min(
    configuracao.camadas_iniciais_congeladas,
    quantidade_camadas,
)

for camada in modelo.bert.encoder.layer[:quantidade_congelada]:
    for parametro in camada.parameters():
        parametro.requires_grad = False

parametros_totais = sum(
    parametro.numel()
    for parametro in modelo.parameters()
)

parametros_treinaveis = sum(
    parametro.numel()
    for parametro in modelo.parameters()
    if parametro.requires_grad
)

print("Camadas BERT:", quantidade_camadas)
print("Camadas iniciais congeladas:", quantidade_congelada)
print(
    "Parâmetros totais:",
    f"{parametros_totais:,}".replace(",", "."),
)
print(
    "Parâmetros treináveis:",
    f"{parametros_treinaveis:,}".replace(",", "."),
)
print(
    "Percentual treinável:",
    f"{100 * parametros_treinaveis / parametros_totais:.2f}%",
)


## 8. Métricas


In [ ]:
def sigmoid(logits: np.ndarray) -> np.ndarray:
    logits = np.clip(logits, -30, 30)
    return 1.0 / (1.0 + np.exp(-logits))


def calcular_metricas(
    rotulos_reais: np.ndarray,
    probabilidades: np.ndarray,
    limiares,
) -> dict:
    limiares = np.asarray(limiares)
    previstos = (probabilidades >= limiares).astype(int)

    metricas = {
        "f1_micro": f1_score(
            rotulos_reais,
            previstos,
            average="micro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            rotulos_reais,
            previstos,
            average="macro",
            zero_division=0,
        ),
        "f1_ponderado": f1_score(
            rotulos_reais,
            previstos,
            average="weighted",
            zero_division=0,
        ),
        "precisao_micro": precision_score(
            rotulos_reais,
            previstos,
            average="micro",
            zero_division=0,
        ),
        "revocacao_micro": recall_score(
            rotulos_reais,
            previstos,
            average="micro",
            zero_division=0,
        ),
        "acuracia_exata": accuracy_score(
            rotulos_reais,
            previstos,
        ),
        "hamming_loss": hamming_loss(
            rotulos_reais,
            previstos,
        ),
    }

    try:
        metricas["roc_auc_macro"] = roc_auc_score(
            rotulos_reais,
            probabilidades,
            average="macro",
        )
    except ValueError:
        metricas["roc_auc_macro"] = np.nan

    return {
        chave: float(valor)
        for chave, valor in metricas.items()
    }


def metricas_treinamento(predicao) -> dict:
    logits = predicao.predictions
    rotulos_reais = predicao.label_ids

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilidades = sigmoid(logits)

    return calcular_metricas(
        rotulos_reais.astype(int),
        probabilidades,
        configuracao.limiar_inicial,
    )


## 9. Treinador multirrótulo com suavização leve


In [ ]:
class TreinadorMultirrotulo(Trainer):
    def __init__(
        self,
        pesos_positivos: torch.Tensor,
        suavizacao_rotulos: float,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.pesos_positivos = pesos_positivos
        self.suavizacao_rotulos = suavizacao_rotulos

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        rotulos = inputs["labels"].float()
        entradas = {
            chave: valor
            for chave, valor in inputs.items()
            if chave != "labels"
        }

        if self.suavizacao_rotulos > 0:
            rotulos = (
                rotulos * (1.0 - self.suavizacao_rotulos)
                + 0.5 * self.suavizacao_rotulos
            )

        saidas = model(**entradas)
        logits = saidas.logits

        funcao_perda = torch.nn.BCEWithLogitsLoss(
            pos_weight=self.pesos_positivos.to(logits.device)
        )

        perda = funcao_perda(logits, rotulos)

        if return_outputs:
            return perda, saidas

        return perda


## 10. Configurar treinamento e retomada automática


In [ ]:
argumentos_treinamento = TrainingArguments(
    output_dir=str(PASTA_CHECKPOINTS),
    num_train_epochs=configuracao.epocas,
    learning_rate=configuracao.taxa_aprendizado,
    per_device_train_batch_size=configuracao.lote_treino,
    per_device_eval_batch_size=configuracao.lote_avaliacao,
    gradient_accumulation_steps=configuracao.acumulacao_gradientes,
    weight_decay=configuracao.decaimento_peso,
    warmup_ratio=configuracao.aquecimento,
    max_grad_norm=configuracao.maximo_gradiente,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=configuracao.passos_log,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    save_safetensors=True,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=configuracao.semente,
    data_seed=configuracao.semente,
    dataloader_num_workers=2,
    dataloader_pin_memory=torch.cuda.is_available(),
    remove_unused_columns=True,
)


treinador = TreinadorMultirrotulo(
    pesos_positivos=tensor_pesos_positivos,
    suavizacao_rotulos=configuracao.suavizacao_rotulos,
    model=modelo,
    args=argumentos_treinamento,
    train_dataset=datasets_tokenizados["treino"],
    eval_dataset=datasets_tokenizados["validacao"],
    processing_class=tokenizador,
    data_collator=colator,
    compute_metrics=metricas_treinamento,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=configuracao.paciencia
        )
    ],
)

ultimo_checkpoint = None

if configuracao.retomar_checkpoint and PASTA_CHECKPOINTS.exists():
    ultimo_checkpoint = get_last_checkpoint(str(PASTA_CHECKPOINTS))

if ultimo_checkpoint:
    print("Treinamento será retomado de:", ultimo_checkpoint)
else:
    print("Novo treinamento será iniciado.")


## 11. Treinar


In [ ]:
resultado_treinamento = treinador.train(
    resume_from_checkpoint=ultimo_checkpoint
)

print("Treinamento concluído.")
display(pd.DataFrame([resultado_treinamento.metrics]).transpose())


## 12. Salvar imediatamente o melhor modelo


In [ ]:
treinador.save_model(str(PASTA_MODELO))
tokenizador.save_pretrained(str(PASTA_MODELO))

modelo_final = treinador.model
modelo_final.config.id2label = ID_PARA_ROTULO
modelo_final.config.label2id = ROTULO_PARA_ID
modelo_final.config.problem_type = "multi_label_classification"
modelo_final.config.save_pretrained(str(PASTA_MODELO))

print("Melhor modelo salvo em:", PASTA_MODELO)


## 13. Histórico do treinamento


In [ ]:
historico = pd.DataFrame(treinador.state.log_history)

display(historico)

historico.to_csv(
    PASTA_METRICAS / "historico_treinamento.csv",
    index=False,
)

perdas_treino = historico[
    historico.get("loss", pd.Series(dtype=float)).notna()
][["step", "loss"]]

if not perdas_treino.empty:
    plt.figure(figsize=(10, 5))
    plt.plot(perdas_treino["step"], perdas_treino["loss"])
    plt.xlabel("Passo")
    plt.ylabel("Perda")
    plt.title("Perda durante o treinamento")
    plt.grid()
    plt.show()

avaliacoes = historico[
    historico.get("eval_loss", pd.Series(dtype=float)).notna()
].copy()

if not avaliacoes.empty:
    plt.figure(figsize=(8, 5))
    plt.plot(
        avaliacoes["epoch"],
        avaliacoes["eval_loss"],
        marker="o",
        label="perda de validação",
    )
    plt.xlabel("Época")
    plt.ylabel("Perda")
    plt.title("Perda de validação")
    plt.legend()
    plt.grid()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(
        avaliacoes["epoch"],
        avaliacoes["eval_f1_micro"],
        marker="o",
        label="F1 micro",
    )
    plt.plot(
        avaliacoes["epoch"],
        avaliacoes["eval_f1_macro"],
        marker="o",
        label="F1 macro",
    )
    plt.xlabel("Época")
    plt.ylabel("F1")
    plt.title("F1 na validação")
    plt.legend()
    plt.grid()
    plt.show()


## 14. Otimizar limiares na validação


In [ ]:
predicao_validacao = treinador.predict(
    datasets_tokenizados["validacao"]
)

logits_validacao = predicao_validacao.predictions

if isinstance(logits_validacao, tuple):
    logits_validacao = logits_validacao[0]

probabilidades_validacao = sigmoid(logits_validacao)
rotulos_validacao = np.asarray(
    tabelas["validacao"]["labels"].tolist(),
    dtype=int,
)


def encontrar_melhor_limiar(
    reais: np.ndarray,
    probabilidades: np.ndarray,
) -> tuple[float, float]:
    melhor_limiar = configuracao.limiar_inicial
    melhor_f1 = -1.0

    candidatos = np.arange(
        configuracao.limiar_minimo,
        configuracao.limiar_maximo
        + configuracao.passo_limiar,
        configuracao.passo_limiar,
    )

    for limiar in candidatos:
        previstos = (probabilidades >= limiar).astype(int)
        f1 = f1_score(
            reais,
            previstos,
            zero_division=0,
        )

        melhorou = f1 > melhor_f1 + 1e-6
        empatou = abs(f1 - melhor_f1) <= 1e-6
        mais_proximo_de_meio = (
            abs(limiar - configuracao.limiar_inicial)
            < abs(melhor_limiar - configuracao.limiar_inicial)
        )

        if melhorou or (empatou and mais_proximo_de_meio):
            melhor_f1 = f1
            melhor_limiar = float(limiar)

    return melhor_limiar, melhor_f1


limiares_otimizados = []
resultados_limiares = []

for indice, rotulo in enumerate(ROTULOS):
    limiar, f1 = encontrar_melhor_limiar(
        rotulos_validacao[:, indice],
        probabilidades_validacao[:, indice],
    )

    limiares_otimizados.append(limiar)
    resultados_limiares.append(
        {
            "rotulo": rotulo,
            "limiar": limiar,
            "f1_validacao": f1,
        }
    )


tabela_limiares = pd.DataFrame(resultados_limiares)
display(tabela_limiares)

tabela_limiares.to_csv(
    PASTA_METRICAS / "limiares_otimizados.csv",
    index=False,
)

(PASTA_MODELO / "limiares.json").write_text(
    json.dumps(
        {
            rotulo: float(limiar)
            for rotulo, limiar in zip(
                ROTULOS,
                limiares_otimizados,
            )
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


## 15. Avaliar no teste sintético


In [ ]:
def obter_probabilidades(nome_split: str) -> tuple[np.ndarray, np.ndarray]:
    predicao = treinador.predict(
        datasets_tokenizados[nome_split]
    )

    logits = predicao.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    probabilidades = sigmoid(logits)
    reais = np.asarray(
        tabelas[nome_split]["labels"].tolist(),
        dtype=int,
    )

    return probabilidades, reais


probabilidades_teste, rotulos_teste = obter_probabilidades("teste")

previstos_padrao = (
    probabilidades_teste >= configuracao.limiar_inicial
).astype(int)

previstos_otimizados = (
    probabilidades_teste >= np.asarray(limiares_otimizados)
).astype(int)

metricas_padrao = calcular_metricas(
    rotulos_teste,
    probabilidades_teste,
    configuracao.limiar_inicial,
)

metricas_otimizadas = calcular_metricas(
    rotulos_teste,
    probabilidades_teste,
    limiares_otimizados,
)

comparacao_metricas = pd.DataFrame(
    [
        {
            "configuracao": "limiar_0.50",
            **metricas_padrao,
        },
        {
            "configuracao": "limiares_otimizados",
            **metricas_otimizadas,
        },
    ]
)

display(comparacao_metricas)

comparacao_metricas.to_csv(
    PASTA_METRICAS / "comparacao_metricas_teste.csv",
    index=False,
)


## 16. Métricas por classe, matrizes e erros


In [ ]:
relatorio_classes = classification_report(
    rotulos_teste,
    previstos_otimizados,
    target_names=ROTULOS,
    output_dict=True,
    zero_division=0,
)

tabela_classes = pd.DataFrame(relatorio_classes).transpose()
display(tabela_classes)

tabela_classes.to_csv(
    PASTA_METRICAS / "metricas_teste_por_classe.csv"
)

matrizes = multilabel_confusion_matrix(
    rotulos_teste,
    previstos_otimizados,
)

for indice, rotulo in enumerate(ROTULOS):
    matriz = matrizes[indice]

    plt.figure(figsize=(5, 5))
    plt.imshow(matriz)
    plt.xticks([0, 1], ["previsto 0", "previsto 1"])
    plt.yticks([0, 1], ["real 0", "real 1"])
    plt.title(f"Matriz de confusão — {rotulo}")
    plt.colorbar()

    for linha in range(2):
        for coluna in range(2):
            plt.text(
                coluna,
                linha,
                int(matriz[linha, coluna]),
                ha="center",
                va="center",
            )

    plt.tight_layout()
    plt.show()

colunas_contexto = [
    coluna
    for coluna in (
        "id",
        "texto",
        "perfil",
        "registro",
        "regiao",
    )
    if coluna in tabelas["teste"].columns
]

tabela_erros = tabelas["teste"][colunas_contexto].copy()

for indice, rotulo in enumerate(ROTULOS):
    tabela_erros[f"real_{rotulo}"] = rotulos_teste[:, indice]
    tabela_erros[f"prob_{rotulo}"] = probabilidades_teste[:, indice]
    tabela_erros[f"prev_{rotulo}"] = previstos_otimizados[:, indice]


tabela_erros["quantidade_erros"] = np.sum(
    rotulos_teste != previstos_otimizados,
    axis=1,
)

erros = tabela_erros[
    tabela_erros["quantidade_erros"] > 0
].sort_values(
    "quantidade_erros",
    ascending=False,
)

print("Exemplos com pelo menos um erro:", len(erros))
display(erros.head(30))

erros.to_csv(
    PASTA_METRICAS / "erros_teste.csv",
    index=False,
)


## 17. Avaliar em teste externo, quando existir


In [ ]:
metricas_externas = None

if "teste_externo" in datasets_tokenizados:
    probabilidades_externas, rotulos_externos = obter_probabilidades(
        "teste_externo"
    )

    previstos_externos = (
        probabilidades_externas >= np.asarray(limiares_otimizados)
    ).astype(int)

    metricas_externas = calcular_metricas(
        rotulos_externos,
        probabilidades_externas,
        limiares_otimizados,
    )

    print("Métricas no teste externo:")
    display(pd.DataFrame([metricas_externas]).transpose())

    relatorio_externo = pd.DataFrame(
        classification_report(
            rotulos_externos,
            previstos_externos,
            target_names=ROTULOS,
            output_dict=True,
            zero_division=0,
        )
    ).transpose()

    display(relatorio_externo)

    relatorio_externo.to_csv(
        PASTA_METRICAS / "metricas_teste_externo_por_classe.csv"
    )

    pd.DataFrame([metricas_externas]).to_csv(
        PASTA_METRICAS / "metricas_teste_externo.csv",
        index=False,
    )
else:
    print(
        "teste_externo.jsonl não encontrado. "
        "O treinamento funciona, mas a generalização real ainda não foi medida."
    )


## 18. Testar frases manualmente


In [ ]:
def classificar_textos(textos: list[str]) -> pd.DataFrame:
    modelo_avaliacao = treinador.model
    modelo_avaliacao.eval()

    entradas = tokenizador(
        textos,
        padding=True,
        truncation=True,
        max_length=configuracao.comprimento_maximo,
        return_tensors="pt",
    )

    dispositivo = next(modelo_avaliacao.parameters()).device
    entradas = {
        chave: valor.to(dispositivo)
        for chave, valor in entradas.items()
    }

    with torch.no_grad():
        logits = modelo_avaliacao(**entradas).logits

    probabilidades = torch.sigmoid(logits).cpu().numpy()
    linhas = []

    for texto, probabilidades_texto in zip(
        textos,
        probabilidades,
    ):
        linha = {"texto": texto}
        ativos = []

        for rotulo, probabilidade, limiar in zip(
            ROTULOS,
            probabilidades_texto,
            limiares_otimizados,
        ):
            linha[f"prob_{rotulo}"] = float(probabilidade)

            if probabilidade >= limiar:
                ativos.append(rotulo)

        linha["rotulos_previstos"] = ativos
        linhas.append(linha)

    return pd.DataFrame(linhas)


frases_teste = [
    "socorro ele está me batendo",
    "ele disse que vai me matar e está com uma faca",
    "comprei uma faca para cortar carne",
    "esse jogo está bom pra caralho",
    "mana me acode ele não solta meu braço",
    "o filme mostrou uma cena de tiro",
    "não desliga ele tá vindo pra cima de mim",
    "o barulho da televisão pareceu um tiro",
]

display(classificar_textos(frases_teste))


## 19. Salvar resultados e criar ZIP final


In [ ]:
predicoes_teste = tabelas["teste"][["id", "texto"]].copy()

for indice, rotulo in enumerate(ROTULOS):
    predicoes_teste[f"real_{rotulo}"] = rotulos_teste[:, indice]
    predicoes_teste[f"prob_{rotulo}"] = probabilidades_teste[:, indice]
    predicoes_teste[f"prev_{rotulo}"] = previstos_otimizados[:, indice]

predicoes_teste.to_csv(
    PASTA_METRICAS / "predicoes_teste.csv",
    index=False,
)

resumo = {
    "configuracao": asdict(configuracao),
    "rotulos": ROTULOS,
    "caminhos": {
        "raiz": str(PASTA_RAIZ),
        "modelo": str(PASTA_MODELO),
        "metricas": str(PASTA_METRICAS),
        "checkpoints": str(PASTA_CHECKPOINTS),
    },
    "parametros": {
        "totais": parametros_totais,
        "treinaveis": parametros_treinaveis,
    },
    "limiares": {
        rotulo: float(limiar)
        for rotulo, limiar in zip(
            ROTULOS,
            limiares_otimizados,
        )
    },
    "pesos_positivos": {
        rotulo: float(peso)
        for rotulo, peso in zip(
            ROTULOS,
            pesos_positivos,
        )
    },
    "metricas_limiar_0_50": metricas_padrao,
    "metricas_limiares_otimizados": metricas_otimizadas,
    "metricas_teste_externo": metricas_externas,
}

(PASTA_METRICAS / "resumo_experimento.json").write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

pasta_entrega = PASTA_RESULTADOS / "entrega_bertimbau_regularizado"

if pasta_entrega.exists():
    shutil.rmtree(pasta_entrega)

pasta_entrega.mkdir(parents=True)

shutil.copytree(
    PASTA_MODELO,
    pasta_entrega / "modelo",
)

shutil.copytree(
    PASTA_METRICAS,
    pasta_entrega / "metricas",
)

arquivo_zip = shutil.make_archive(
    str(PASTA_RESULTADOS / "modelo_nlp_regularizado"),
    "zip",
    root_dir=str(pasta_entrega),
)

print("Modelo salvo em:", PASTA_MODELO)
print("Métricas salvas em:", PASTA_METRICAS)
print("ZIP final sem checkpoints:", arquivo_zip)


## Resultado esperado no projeto

```text
models/nlp/
├── bertimbau_multirrotulo_regularizado/
└── checkpoints_bertimbau_regularizado/

outputs/metrics/nlp/bertimbau_regularizado/
├── auditoria_vazamento.csv
├── historico_treinamento.csv
├── limiares_otimizados.csv
├── comparacao_metricas_teste.csv
├── metricas_teste_por_classe.csv
├── erros_teste.csv
├── predicoes_teste.csv
└── resumo_experimento.json

outputs/resultados_nlp/
└── modelo_nlp_regularizado.zip
```

Os checkpoints permitem continuar um treinamento interrompido. O ZIP final não contém os checkpoints, por isso é menor.
